# ⭐ Feature Engineering for Telecom Customer Churn Prediction
Creating Powerful Features for Better Model Performance

---

## 1. Introduction & Business Context

Customer churn is one of the most critical challenges in the telecom industry. Acquiring a new customer costs **5–25x more** than retaining an existing one. The goal of this notebook is to engineer powerful, predictive features that help build a high-accuracy churn prediction model.

### Why Feature Engineering Matters for Churn Prediction
- **Raw data is rarely enough**: Customer IDs, dates, and raw usage numbers don't capture behavioral patterns.
- **Domain knowledge drives performance**: Understanding *why* customers leave (high bills, poor service, low engagement) allows us to encode those signals into features.
- **Better features > better algorithms**: A simple model with excellent features often outperforms a complex model with poor features.

In this notebook, we will systematically transform raw telecom data into a rich feature set designed to maximize model performance.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML utilities for feature importance preview
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")

## 2. Dataset Overview

This telecom churn dataset contains the following columns:
- `customer_id`: Unique identifier
- `signup_date`: When the customer joined
- `age`, `gender`, `education_level`, `employment_status`: Demographics
- `city`, `monthly_income`: Geographic and income attributes
- `monthly_bill`, `internet_usage_gb`, `call_minutes`: Usage & billing
- `contract_type`: `Monthly`, `6-Month`, `12-Month`
- `support_tickets`: Number of support tickets raised
- `customer_feedback`: Free-text feedback
- `churn`: Target variable (1 = churned, 0 = active)


In [ ]:
df = pd.read_csv("telecom_customer_churn.csv", parse_dates=['signup_date'])

df.head()

In [ ]:
# Basic info
print(f"Dataset shape: {df.shape}")
print(f"\nChurn rate: {df['churn'].mean():.2%}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.describe()

## 3. Feature Engineering Strategy

Our approach follows a structured pipeline:

1. **Temporal Features** → Capture customer lifecycle and seasonality
2. **Usage & Behavior Features** → Encode engagement and value patterns
3. **Customer Profile Features** → Segment and encode demographics
4. **Contract & Loyalty Features** → Capture commitment and satisfaction trends
5. **Feedback & Sentiment Features** → Extract signals from unstructured text
6. **Risk & Behavioral Flags** → Create composite risk indicators
7. **Feature Selection** → Filter for the most predictive signals

Each section includes **"Why this feature helps"** explanations to connect engineering decisions to business intuition.

---
## 4. Temporal & Date Features

Dates contain rich information about customer maturity, seasonality, and lifecycle stage.

In [ ]:
# Reference date (e.g., today or data extraction date)
reference_date = datetime(2024, 4, 1)

# Customer tenure in days and months
df['tenure_days'] = (reference_date - df['signup_date']).dt.days
df['tenure_months'] = (df['tenure_days'] / 30.44).astype(int)  # Average days per month

# Season/Quarter of signup
df['signup_month'] = df['signup_date'].dt.month
df['signup_quarter'] = df['signup_date'].dt.quarter
df['signup_season'] = df['signup_month'].map({12: 'Winter', 1: 'Winter', 2: 'Winter',
                                               3: 'Spring', 4: 'Spring', 5: 'Spring',
                                               6: 'Summer', 7: 'Summer', 8: 'Summer',
                                               9: 'Fall', 10: 'Fall', 11: 'Fall'})

# Weekend vs weekday signup (possible impulse vs planned decision)
df['signup_weekday'] = df['signup_date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['signup_is_weekend'] = (df['signup_weekday'] >= 5).astype(int)

print("Temporal features created:")
print(df[['signup_date', 'tenure_days', 'tenure_months', 'signup_month', 'signup_season', 'signup_is_weekend']].head())

**Why these features help:**
- **`tenure_months`**: Long-tenured customers often have higher switching costs and are less likely to churn.
- **`signup_season`**: Customers who sign up during promotional seasons (e.g., holidays) may have different loyalty profiles.
- **`signup_is_weekend`**: Weekend signups might indicate impulse decisions, which could correlate with higher churn.

In [ ]:
# Visualize tenure distribution by churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='tenure_months', hue='churn', bins=30, kde=True, ax=axes[0])
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_xlabel('Tenure (Months)')

churn_by_season = df.groupby('signup_season')['churn'].mean().reindex(['Spring', 'Summer', 'Fall', 'Winter'])
churn_by_season.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Churn Rate by Signup Season')
axes[1].set_ylabel('Churn Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 5. Usage & Behavior Features

These features capture how customers interact with services and their perceived value.

In [ ]:
# ARPU is already monthly_bill, but let's create normalized versions
df['arpu'] = df['monthly_bill']

# Usage intensity: Data per call minute (engagement density)
# Handle division by zero
df['usage_intensity_gb_per_min'] = df['internet_usage_gb'] / (df['call_minutes'] + 1)

# Data usage per dollar (value perception)
df['data_per_dollar'] = df['internet_usage_gb'] / (df['monthly_bill'] + 0.01)

# Bill-to-income ratio (affordability stress)
df['bill_to_income_ratio'] = df['monthly_bill'] / (df['monthly_income'] + 1)

# High/Low usage flags based on percentiles
data_median = df['internet_usage_gb'].median()
bill_median = df['monthly_bill'].median()

df['high_data_user'] = (df['internet_usage_gb'] > data_median).astype(int)
df['high_bill_flag'] = (df['monthly_bill'] > bill_median).astype(int)
df['low_usage_flag'] = (df['internet_usage_gb'] < df['internet_usage_gb'].quantile(0.25)).astype(int)

# Total engagement score
df['total_engagement'] = df['internet_usage_gb'] + (df['call_minutes'] / 10)  # Normalize scale

print("Usage & behavior features created:")
print(df[['monthly_bill', 'arpu', 'usage_intensity_gb_per_min', 'data_per_dollar', 
          'bill_to_income_ratio', 'high_data_user', 'low_usage_flag']].head())

**Why these features help:**
- **`arpu`**: Direct revenue indicator; low ARPU customers may not be worth retaining, while declining ARPU signals disengagement.
- **`data_per_dollar`**: Customers getting poor value (low GB per dollar) are price-sensitive and more likely to churn.
- **`bill_to_income_ratio`**: Financial stress indicator; high ratios suggest affordability-driven churn.
- **`usage_intensity`**: Heavy data users who rarely call may be broadband-only customers with different loyalty patterns.
- **`low_usage_flag`**: Disengagement is a leading indicator of churn.

In [ ]:
# Visualize key usage features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='churn', y='data_per_dollar', ax=axes[0,0])
axes[0,0].set_title('Data per Dollar by Churn Status')
axes[0,0].set_ylim(0, df['data_per_dollar'].quantile(0.95))

sns.boxplot(data=df, x='churn', y='bill_to_income_ratio', ax=axes[0,1])
axes[0,1].set_title('Bill-to-Income Ratio by Churn Status')
axes[0,1].set_ylim(0, df['bill_to_income_ratio'].quantile(0.95))

sns.countplot(data=df, x='high_data_user', hue='churn', ax=axes[1,0])
axes[1,0].set_title('Churn by High Data User Flag')
axes[1,0].set_xticklabels(['Low/Med Data', 'High Data'])

sns.countplot(data=df, x='low_usage_flag', hue='churn', ax=axes[1,1])
axes[1,1].set_title('Churn by Low Usage Flag')
axes[1,1].set_xticklabels(['Normal/High Usage', 'Low Usage'])

plt.tight_layout()
plt.show()

---
## 6. Customer Profile Features

Demographic and geographic segmentation helps the model generalize across customer segments.

In [ ]:
# Age groups
bins = [0, 25, 35, 50, 65, 100]
labels = ['18-25', '26-35', '36-50', '51-65', '65+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

# Income segments
income_q = df['monthly_income'].quantile([0.33, 0.67])
df['income_segment'] = pd.cut(df['monthly_income'], 
                               bins=[0, income_q[0.33], income_q[0.67], float('inf')],
                               labels=['Low', 'Medium', 'High'])

# Education + Employment interaction
df['edu_employment'] = df['education_level'].astype(str) + '_' + df['employment_status'].astype(str)

# City churn rate encoding (target encoding with smoothing)
# Calculate mean churn rate per city
city_churn = df.groupby('city')['churn'].agg(['mean', 'count'])
global_mean = df['churn'].mean()
smoothing_factor = 10

# Smoothed target encoding to prevent overfitting
city_churn['smoothed_churn_rate'] = (
    (city_churn['count'] * city_churn['mean'] + smoothing_factor * global_mean) /
    (city_churn['count'] + smoothing_factor)
)

df['city_churn_rate'] = df['city'].map(city_churn['smoothed_churn_rate'])

# Derive region from city since the CSV does not include a region column
city_region_map = {
    'Karachi': 'Sindh', 'Hyderabad': 'Sindh',
    'Lahore': 'Punjab', 'Faisalabad': 'Punjab', 'Rawalpindi': 'Punjab',
    'Gujranwala': 'Punjab', 'Multan': 'Punjab', 'Sialkot': 'Punjab',
    'Peshawar': 'KPK', 'Quetta': 'Balochistan', 'Islamabad': 'Islamabad Capital Territory'
}
df['region'] = df['city'].map(city_region_map).fillna('Other')
df['region_encoded'] = pd.Categorical(df['region']).codes

# High churn city flag (cities with above-average churn)
high_churn_cities = city_churn[city_churn['smoothed_churn_rate'] > global_mean].index.tolist()
df['high_churn_city'] = df['city'].isin(high_churn_cities).astype(int)

print("Customer profile features created:")
print(df[['age', 'age_group', 'monthly_income', 'income_segment', 'edu_employment', 'city_churn_rate', 'high_churn_city']].head())

**Why these features help:**
- **`age_group`**: Younger customers often have lower switching costs; seniors may prefer stability.
- **`income_segment`**: Income affects price sensitivity and service tier preferences.
- **`edu_employment`**: Interaction terms capture nuanced segments (e.g., unemployed PhDs vs. employed high school graduates).
- **`city_churn_rate`**: Geographic churn patterns reflect local competition, infrastructure quality, and regional service issues.
- **`high_churn_city`**: Binary flag makes it easy for tree-based models to split on problematic regions.

In [ ]:
# Visualize profile features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

churn_by_age = df.groupby('age_group')['churn'].mean()
churn_by_age.plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Churn Rate by Age Group')
axes[0,0].tick_params(axis='x', rotation=45)

churn_by_income = df.groupby('income_segment')['churn'].mean()
churn_by_income.plot(kind='bar', ax=axes[0,1], color='darkgreen')
axes[0,1].set_title('Churn Rate by Income Segment')
axes[0,1].tick_params(axis='x', rotation=0)

sns.boxplot(data=df, x='churn', y='city_churn_rate', ax=axes[1,0])
axes[1,0].set_title('City Churn Rate Distribution by Customer Churn')

sns.countplot(data=df, x='high_churn_city', hue='churn', ax=axes[1,1])
axes[1,1].set_title('Churn by High Churn City Flag')
axes[1,1].set_xticklabels(['Low Churn City', 'High Churn City'])

plt.tight_layout()
plt.show()

---
## 7. Contract & Loyalty Features

Contract details and support interactions reveal commitment level and satisfaction trends.

In [ ]:
# Contract type encoding
contract_map = {'Monthly': 0, '6-Month': 1, '12-Month': 2}
df['contract_type_encoded'] = df['contract_type'].map(contract_map)

# Tenure × Contract interaction (long-term customers on month-to-month are high risk)
df['tenure_x_contract'] = df['tenure_months'] * df['contract_type_encoded']

# Inverse contract length (month-to-month gets highest value)
df['contract_risk_score'] = 1 / (df['contract_type_encoded'] + 1)  # 1.0, 0.5, 0.33

# Support ticket rate (tickets per month of tenure)
df['ticket_rate'] = df['support_tickets'] / (df['tenure_months'] + 1)

# High ticket flag
df['high_ticket_flag'] = (df['support_tickets'] >= 3).astype(int)

# Recent ticket flag (if we had ticket dates, we'd use recency; here we use count as proxy)
df['frequent_caller'] = (df['ticket_rate'] > df['ticket_rate'].quantile(0.75)).astype(int)

# Loyalty score (composite: tenure + contract length - tickets)
df['loyalty_score'] = (
    df['tenure_months'] / 12 +  # Years of tenure
    df['contract_type_encoded'] * 6 -  # Contract commitment bonus
    df['ticket_rate'] * 2  # Penalty for frequent issues
)

print("Contract & loyalty features created:")
print(df[['contract_type', 'contract_type_encoded', 'tenure_months', 'tenure_x_contract', 
          'ticket_rate', 'loyalty_score']].head())

**Why these features help:**
- **`contract_type_encoded`**: Monthly customers can leave anytime; longer contracts create retention lock-in.
- **`tenure_x_contract`**: A customer who has been month-to-month for 3 years is very different from one on a 2-year contract for 3 months.
- **`ticket_rate`**: Frequent support calls indicate frustration; normalized by tenure to distinguish new vs. chronic issues.
- **`loyalty_score`**: Composite metric that balances tenure, commitment, and dissatisfaction signals.

In [ ]:
# Visualize contract features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

churn_by_contract = df.groupby('contract_type')['churn'].mean()
churn_by_contract.plot(kind='bar', ax=axes[0,0], color='coral')
axes[0,0].set_title('Churn Rate by Contract Type')
axes[0,0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='churn', y='ticket_rate', ax=axes[0,1])
axes[0,1].set_title('Ticket Rate by Churn Status')
axes[0,1].set_ylim(0, df['ticket_rate'].quantile(0.95))

sns.scatterplot(data=df, x='tenure_months', y='loyalty_score', hue='churn', alpha=0.6, ax=axes[1,0])
axes[1,0].set_title('Loyalty Score vs Tenure')

sns.countplot(data=df, x='high_ticket_flag', hue='churn', ax=axes[1,1])
axes[1,1].set_title('Churn by High Ticket Flag')
axes[1,1].set_xticklabels(['< 3 Tickets', '>= 3 Tickets'])

plt.tight_layout()
plt.show()

---
## 8. Feedback & Sentiment Features

Unstructured text feedback contains powerful churn signals when properly extracted.

In [ ]:
# Text preprocessing
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove punctuation
    return text

df['feedback_clean'] = df['customer_feedback'].apply(clean_text)

# Keyword flags for common churn drivers
negative_keywords = {
    'poor_coverage': ['coverage is poor', 'poor in my location', 'frequent call drops', 'call drops'],
    'billing_issues': ['billing issues occurred multiple times', 'billing issues', 'billing'],
    'slow_internet': ['internet speed is very slow', 'service is unreliable during peak hours', 'slow', 'unreliable'],
    'rude_support': ['customer support was rude', 'rude', 'unhelpful'],
    'price_sensitive': ['too expensive compared to competitors', 'too expensive', 'expensive', 'competitors', 'price', 'cost']
}

for flag_name, keywords in negative_keywords.items():
    pattern = '|'.join(keywords)
    df[f'feedback_{flag_name}'] = df['feedback_clean'].str.contains(pattern, regex=True).astype(int)

# Positive sentiment flag
positive_keywords = ['customer support was helpful', 'network quality is good overall', 'satisfied with the service', 'happy with data packages', 'helpful', 'satisfied', 'happy', 'good']
positive_pattern = '|'.join(positive_keywords)
df['feedback_positive'] = df['feedback_clean'].str.contains(positive_pattern, regex=True).astype(int)

# Negative sentiment score (count of negative flags)
negative_flag_cols = [c for c in df.columns if c.startswith('feedback_') and c not in ['feedback_positive', 'feedback_clean']]
df['negative_sentiment_score'] = df[negative_flag_cols].sum(axis=1)

# Feedback length (longer complaints may indicate more frustration)
df['feedback_length'] = df['customer_feedback'].str.len()

# Composite sentiment: positive - negative
df['net_sentiment'] = df['feedback_positive'] - df['negative_sentiment_score']

print("Feedback & sentiment features created:")
sentiment_cols = ['feedback_clean', 'feedback_poor_coverage', 'feedback_billing_issues', 
                  'feedback_slow_internet', 'feedback_positive', 'negative_sentiment_score', 'net_sentiment']
print(df[sentiment_cols].head())

**Why these features help:**
- **`feedback_billing_issues`**: Price complaints are direct churn predictors.
- **`feedback_slow_internet`**: Service quality issues drive customers to competitors.
- **`negative_sentiment_score`**: Aggregated frustration level; multiple complaints = high risk.
- **`net_sentiment`**: Simple but effective polarity score; negative values strongly predict churn.
- **`feedback_length`**: Longer, detailed complaints often indicate serious dissatisfaction.

In [ ]:
# Visualize sentiment features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sentiment_churn = df.groupby('negative_sentiment_score')['churn'].mean()
sentiment_churn.plot(kind='bar', ax=axes[0,0], color='crimson')
axes[0,0].set_title('Churn Rate by Negative Sentiment Score')
axes[0,0].set_xlabel('Number of Negative Flags')

sns.boxplot(data=df, x='churn', y='net_sentiment', ax=axes[0,1])
axes[0,1].set_title('Net Sentiment by Churn Status')

# Stacked bar of feedback categories
feedback_counts = df[negative_flag_cols + ['feedback_positive']].sum().sort_values()
feedback_counts.plot(kind='barh', ax=axes[1,0], color='teal')
axes[1,0].set_title('Frequency of Feedback Keywords')

sns.countplot(data=df, x='feedback_positive', hue='churn', ax=axes[1,1])
axes[1,1].set_title('Churn by Positive Feedback Flag')
axes[1,1].set_xticklabels(['No Positive Words', 'Has Positive Words'])

plt.tight_layout()
plt.show()

---
## 9. Risk & Behavioral Flags

Composite flags combine multiple signals into high-impact binary indicators.

In [ ]:
# High bill + Low usage flag (paying a lot but not using → poor value perception)
df['high_bill_low_usage'] = ((df['high_bill_flag'] == 1) & (df['low_usage_flag'] == 1)).astype(int)

# Many support tickets flag (chronic dissatisfaction)
df['many_tickets'] = (df['support_tickets'] >= 4).astype(int)

# Recent high churn risk (new customer + month-to-month + high bill)
df['new_high_risk'] = ((df['tenure_months'] <= 3) & 
                        (df['contract_type'] == 'Monthly') & 
                        (df['high_bill_flag'] == 1)).astype(int)

# Declining engagement proxy (high bill but low data per dollar)
df['poor_value_perception'] = ((df['data_per_dollar'] < df['data_per_dollar'].quantile(0.25)) & 
                                (df['high_bill_flag'] == 1)).astype(int)

# Multi-issue customer (billing + service + support)
df['multi_issue_customer'] = (
    (df['feedback_billing_issues'] == 1) & 
    ((df['feedback_slow_internet'] == 1) | (df['feedback_poor_coverage'] == 1)) & 
    (df['high_ticket_flag'] == 1)
).astype(int)

# At-risk score (sum of all risk flags)
risk_flags = ['high_bill_low_usage', 'many_tickets', 'new_high_risk', 
              'poor_value_perception', 'multi_issue_customer', 'frequent_caller']
df['at_risk_score'] = df[risk_flags].sum(axis=1)

# Critical risk flag (3+ risk indicators)
df['critical_risk_flag'] = (df['at_risk_score'] >= 3).astype(int)

print("Risk & behavioral flags created:")
risk_cols = ['high_bill_low_usage', 'many_tickets', 'new_high_risk', 
             'poor_value_perception', 'multi_issue_customer', 'at_risk_score', 'critical_risk_flag']
print(df[risk_cols].head(10))

**Why these features help:**
- **`high_bill_low_usage`**: Classic value mismatch; these customers feel they're overpaying.
- **`new_high_risk`**: First 3 months are critical; new month-to-month customers with high bills churn fastest.
- **`multi_issue_customer`**: Customers with billing, service, AND support issues are almost certain to leave.
- **`at_risk_score`**: Cumulative risk index; models can learn threshold effects (2+ flags = danger zone).
- **`critical_risk_flag`**: Binary flag for immediate intervention by retention teams.

In [ ]:
# Visualize risk flags
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

risk_churn = df.groupby('at_risk_score')['churn'].mean()
risk_churn.plot(kind='bar', ax=axes[0,0], color='darkred')
axes[0,0].set_title('Churn Rate by At-Risk Score')
axes[0,0].set_xlabel('Number of Risk Flags')

sns.countplot(data=df, x='critical_risk_flag', hue='churn', ax=axes[0,1])
axes[0,1].set_title('Churn by Critical Risk Flag')
axes[0,1].set_xticklabels(['< 3 Risk Flags', '>= 3 Risk Flags'])

# Risk flag co-occurrence heatmap
risk_corr = df[risk_flags].corr()
sns.heatmap(risk_corr, annot=True, cmap='coolwarm', center=0, ax=axes[1,0])
axes[1,0].set_title('Risk Flag Correlation Matrix')

# Churn by new high risk
sns.countplot(data=df, x='new_high_risk', hue='churn', ax=axes[1,1])
axes[1,1].set_title('Churn by New High Risk Flag')
axes[1,1].set_xticklabels(['Not New High Risk', 'New High Risk'])

plt.tight_layout()
plt.show()

---
## 10. Feature Selection & Final Feature Set

Now we evaluate which features are most predictive and build our final feature matrix.

In [ ]:
# Define feature lists
numeric_features = [
    'tenure_days', 'tenure_months', 'signup_month', 'signup_quarter', 'signup_weekday', 'signup_is_weekend',
    'age', 'monthly_income', 'monthly_bill', 'internet_usage_gb', 'call_minutes',
    'arpu', 'usage_intensity_gb_per_min', 'data_per_dollar', 'bill_to_income_ratio',
    'high_data_user', 'high_bill_flag', 'low_usage_flag', 'total_engagement',
    'city_churn_rate', 'high_churn_city', 'region_encoded',
    'contract_type_encoded', 'tenure_x_contract', 'contract_risk_score',
    'ticket_rate', 'high_ticket_flag', 'frequent_caller', 'loyalty_score',
    'feedback_poor_coverage', 'feedback_billing_issues', 'feedback_slow_internet',
    'feedback_rude_support', 'feedback_price_sensitive', 'feedback_positive',
    'negative_sentiment_score', 'feedback_length', 'net_sentiment',
    'high_bill_low_usage', 'many_tickets', 'new_high_risk',
    'poor_value_perception', 'multi_issue_customer', 'at_risk_score', 'critical_risk_flag'
]

# Add encoded categoricals
categorical_dummies = pd.get_dummies(df[['gender', 'age_group', 'income_segment', 'signup_season']], drop_first=True)

# Final feature matrix
X = pd.concat([df[numeric_features], categorical_dummies], axis=1)
y = df['churn']

print(f"Total features: {X.shape[1]}")
print(f"Training samples: {X.shape[0]}")
print("\nFeature names:")
print(list(X.columns))

In [ ]:
# Correlation with churn
correlations = X.corrwith(y).sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 12))
correlations.plot(kind='barh', color=['crimson' if v > 0 else 'steelblue' for v in correlations])
plt.title('Feature Correlation with Churn')
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Top 10 features by absolute correlation with churn:")
print(correlations.head(10))

In [ ]:
# Feature importance preview using Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X_train, y_train)

# Feature importance
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 12))
importance.head(20).plot(kind='barh', color='forestgreen')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 15 features by Random Forest importance:")
print(importance.head(15))

---
## 11. Before vs After Feature Engineering Comparison

Let's compare model performance using only raw features versus our engineered feature set.

In [ ]:
# Baseline features (raw data only)
baseline_features = ['age', 'monthly_income', 'monthly_bill', 'internet_usage_gb', 
                     'call_minutes', 'support_tickets']
X_baseline = df[baseline_features]

# Encode categoricals for baseline
baseline_cats = pd.get_dummies(df[['gender', 'contract_type']], drop_first=True)
X_baseline = pd.concat([X_baseline, baseline_cats], axis=1)

# Split baseline
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42, stratify=y)

# Split engineered
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Train models
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_engineered = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)

rf_baseline.fit(Xb_train, yb_train)
rf_engineered.fit(Xe_train, ye_train)

# Predictions
yb_pred = rf_baseline.predict(Xb_test)
yb_proba = rf_baseline.predict_proba(Xb_test)[:, 1]

ye_pred = rf_engineered.predict(Xe_test)
ye_proba = rf_engineered.predict_proba(Xe_test)[:, 1]

# Metrics
baseline_acc = accuracy_score(yb_test, yb_pred)
baseline_auc = roc_auc_score(yb_test, yb_proba)

engineered_acc = accuracy_score(ye_test, ye_pred)
engineered_auc = roc_auc_score(ye_test, ye_proba)

print("=" * 50)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 50)
print(f"\n{'Metric':<20} {'Baseline':<15} {'Engineered':<15} {'Improvement':<15}")
print("-" * 65)
print(f"{'Accuracy':<20} {baseline_acc:<15.4f} {engineered_acc:<15.4f} {engineered_acc - baseline_acc:+.4f}")
print(f"{'ROC-AUC':<20} {baseline_auc:<15.4f} {engineered_auc:<15.4f} {engineered_auc - baseline_auc:+.4f}")
print(f"\n{'Feature Count':<20} {X_baseline.shape[1]:<15} {X.shape[1]:<15} {X.shape[1] - X_baseline.shape[1]:+d}")
print("=" * 50)

In [ ]:
# Visualization of improvement
metrics = ['Accuracy', 'ROC-AUC']
baseline_scores = [baseline_acc, baseline_auc]
engineered_scores = [engineered_acc, engineered_auc]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline (Raw Features)', color='lightcoral')
bars2 = ax.bar(x + width/2, engineered_scores, width, label='Engineered Features', color='mediumseagreen')

ax.set_ylabel('Score')
ax.set_title('Model Performance: Baseline vs. Engineered Features')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.show()

---
## 12. Summary & Recommendations for Modeling

### Feature Engineering Summary

| Category | Key Features Created | Impact |
|----------|---------------------|--------|
| **Temporal** | `tenure_months`, `signup_season`, `signup_is_weekend` | Captures lifecycle stage and acquisition context |
| **Usage & Behavior** | `data_per_dollar`, `bill_to_income_ratio`, `usage_intensity` | Encodes value perception and engagement |
| **Customer Profile** | `age_group`, `income_segment`, `city_churn_rate` | Segments customers by demographic and geographic risk |
| **Contract & Loyalty** | `tenure_x_contract`, `ticket_rate`, `loyalty_score` | Measures commitment and satisfaction trends |
| **Feedback & Sentiment** | `negative_sentiment_score`, `feedback_billing_issues` | Extracts churn signals from unstructured text |
| **Risk Flags** | `at_risk_score`, `critical_risk_flag`, `new_high_risk` | Composite indicators for immediate intervention |

### Key Insights
1. **Tenure and contract type** are consistently among the strongest predictors—customers on month-to-month plans with short tenure are highest risk.
2. **Sentiment features** from text feedback add significant signal beyond structured data.
3. **Composite risk flags** (`at_risk_score`, `critical_risk_flag`) provide actionable thresholds for retention campaigns.
4. **Geographic encoding** (`city_churn_rate`) captures local market effects that raw city names miss.

### Recommendations for Modeling
- **Start with tree-based models** (Random Forest, XGBoost, LightGBM)—they handle the mix of numeric, categorical, and binary flags well.
- **Use `at_risk_score` and `critical_risk_flag`** as primary segmentation variables for retention campaigns.
- **Monitor `data_per_dollar`** and `bill_to_income_ratio` for pricing optimization opportunities.
- **Re-engineer `city_churn_rate`** regularly as market conditions change.
- **Consider SMOTE or class weighting** if churn rate is imbalanced in production data.
- **Deploy feedback sentiment monitoring** in real-time to flag at-risk customers before they call support.

### Next Steps
1. Hyperparameter tuning on the engineered feature set
2. Try gradient boosting (XGBoost/LightGBM/CatBoost) for marginal gains
3. Implement SHAP analysis for model interpretability
4. A/B test retention campaigns using `critical_risk_flag` segmentation

---
*Notebook complete. The engineered feature set is ready for model training and deployment.*